# Lab 5.3 &mdash; Memory, approval and resume

**About 25 minutes** &middot; Day 2 &middot; Module 5 &mdash; LangChain &amp; LangGraph

A **checkpointer** saves the state after every node, filed under a **thread id**. That one idea gives AskOps a memory of the conversation, a record of every step, resume after a crash, and a pause before `open_incident` until a person says yes.

Run the cells in order, with **Shift + Enter**. Under each cell, **You should see** says what to
expect. The model is real, so its words change from run to run. The shape of the result does not.

**The result:** the full AskOps agent, with all four tools, remembers the conversation and opens an incident only after you approve it.

## Step 1 &mdash; Memory: the same thread remembers

Add a checkpointer to the agent from Lab 5.1. Every call with the same `thread_id` continues the same
conversation. A different `thread_id` starts a new one.

In [ ]:
import json
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from askops import get_llm, langchain_tools, trace, reset_incidents, INCIDENTS
import askops

llm = get_llm()
read_tools = langchain_tools()[:3]            # search_runbooks, get_runbook, list_incidents
SYSTEM = ("You are AskOps, an assistant for on-call engineers. Answer only from tool results, "
          "in at most 4 lines. Cite runbook and incident ids.")
agent = create_agent(llm, read_tools, system_prompt=SYSTEM, checkpointer=InMemorySaver())

def ask(thread, text):
    out = agent.invoke({"messages": [("user", text)]},
                       {"configurable": {"thread_id": thread}, "recursion_limit": 12})
    print(f"[{thread}] {text}\n{out['messages'][-1].content}\n")

ask("eng-a", "Payments returns 502 after deploy. Which runbook do I follow?")
ask("eng-a", "Is an incident already open for it?")
ask("eng-b", "Is an incident already open for it?")

**You should see:** engineer A's second question works, because *it* means the payments 502 from the
first turn: the answer names **INC-9001**. Engineer B asks the same words on a new thread, and the
agent does not know what *it* is. You wrote no code to store the history. The checkpointer did it.

## Step 2 &mdash; What the checkpointer saved

`get_state` returns the latest saved state of a thread. `get_state_history` returns every saved state,
newest first. That history is an **audit trail**: a record of what the agent knew at each step, not a
summary the model writes afterwards.

In [ ]:
cfg = {"configurable": {"thread_id": "eng-a"}}
print("messages saved for eng-a:", len(agent.get_state(cfg).values["messages"]))
print("\nstep  next         messages")
for snap in reversed(list(agent.get_state_history(cfg))):
    nxt = ", ".join(snap.next) or "(done)"
    print(f"{snap.metadata.get('step'):>4}  {nxt:<12} {len(snap.values.get('messages', [])):>8}")

**You should see:** a table with one line per saved step, across both turns of engineer A. The number
of messages grows at every step. Every call resends that whole list, as you saw in Module 4.

## Step 3 &mdash; Resume after a crash

This small graph has two nodes. The second one fails the first time, as if the process died there.
Invoke again with `None` and the same `thread_id`: the run continues from the last saved state.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

class Ticket(TypedDict):
    question: str
    findings: Annotated[list, add]

runs = {"read_runbooks": 0, "read_incidents": 0}
def read_runbooks(s):
    runs["read_runbooks"] += 1
    return {"findings": ["runbooks: " + askops.search_runbooks(s["question"])]}
def read_incidents(s):
    runs["read_incidents"] += 1
    if runs["read_incidents"] == 1:
        raise RuntimeError("the process died here")
    return {"findings": [f"open incidents: {len(json.loads(askops.list_incidents()))}"]}

g = StateGraph(Ticket)
g.add_node("read_runbooks", read_runbooks); g.add_node("read_incidents", read_incidents)
g.add_edge(START, "read_runbooks"); g.add_edge("read_runbooks", "read_incidents")
g.add_edge("read_incidents", END)
app = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "crash-demo"}}

try:
    app.invoke({"question": "502 after deploy", "findings": []}, cfg)
except RuntimeError as exc:
    print("crashed:", exc, "| waiting before:", app.get_state(cfg).next)

print("resumed:", app.invoke(None, cfg)["findings"])
print("times each node ran:", runs)

**You should see:** the crash, then a resumed run with both findings, and `read_runbooks` ran only
**once**. The resumed run did not repeat work that was already saved. If the node that crashed opened
an incident, starting again from the beginning would open it twice.

## Step 4 &mdash; Pause before a write, with `interrupt()`

`interrupt(value)` inside a node stops the run, saves the state, and shows `value` to a person.
`Command(resume=...)` continues it, and what you pass becomes the return value of `interrupt`.
Nothing is running while it waits, so the person can answer a minute later or the next morning.

In [ ]:
from langgraph.types import interrupt, Command

class Draft(TypedDict):
    question: str
    answer: str

def file_incident(s):
    draft = {"title": s["question"][:60], "severity": "medium", "service": "reporting",
             "runbook_id": "RB-302"}
    decision = interrupt(draft)                      # pause here, before the write
    if decision != "yes":
        return {"answer": "Not approved. Nothing opened."}
    return {"answer": "Opened " + json.loads(askops.open_incident(**draft))["id"]}

g = StateGraph(Draft)
g.add_node("file_incident", file_incident)
g.add_edge(START, "file_incident"); g.add_edge("file_incident", END)
app = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "disk-1"}}

reset_incidents()
paused = app.invoke({"question": "Disk on the report nodes is at 95 percent"}, cfg)
print("waiting for a person:", paused["__interrupt__"][0].value)
print("incidents while waiting:", len(INCIDENTS))
print(app.invoke(Command(resume="yes"), cfg)["answer"], "| incidents now:", len(INCIDENTS))

**You should see:** the draft incident, **3** incidents while it waits, then `Opened INC-9004` and
**4** incidents. When the run continues, the node runs again from its first line. That is why the
`interrupt` comes **before** `open_incident`, never after it.

## The result &mdash; the full AskOps agent, with approval

Now give the agent all four tools. `HumanInTheLoopMiddleware` puts an `interrupt` in front of
`open_incident` for you, so the three read tools run freely and the write waits for you. Set
`DECISION` to `"approve"` or `"reject"` and run the cell.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

DECISION = "approve"            # try "reject" too

reset_incidents()
agent = create_agent(llm, langchain_tools(), system_prompt=SYSTEM, checkpointer=InMemorySaver(),
                     middleware=[HumanInTheLoopMiddleware(interrupt_on={"open_incident": True})])
cfg = {"configurable": {"thread_id": "eng-c"}, "recursion_limit": 16}

out = agent.invoke({"messages": [("user",
      "Disk on the report nodes is at 95 percent. Find the runbook, check the open incidents, "
      "and open an incident if none covers it.")]}, cfg)

if "__interrupt__" in out:
    for req in out["__interrupt__"][0].value["action_requests"]:
        print("The agent wants to run:", req["name"], req["args"])
    print("Your decision:", DECISION, "\n")
    decision = {"type": DECISION} if DECISION == "approve" else \
               {"type": "reject", "message": "Not approved by the on-call lead."}
    out = agent.invoke(Command(resume={"decisions": [decision]}), cfg)

trace(out)
print("incidents now:", len(INCIDENTS))

**You should see:** the agent searches the runbooks and lists the incidents on its own. Then it
stops, and prints the `open_incident` call it wants to make, for RB-302. With `"approve"`, the
incident is opened: the answer names **INC-9004**, and there are **4** incidents. With `"reject"`,
nothing is opened, and there are still **3**. The `open_incident` ACTION line appears either way: it
is the model's request. With `"reject"`, the tool never ran.

This is the AskOps agent from Day 1, now with a memory, a record of every step, and a person in charge
of the only tool that writes.